# Anaconda与Python第一份实验

第002单元，先修001。原始合成校准数据，与001完全相同。先读lecture.pdf与lab.pdf。本Notebook没有外部文件读取、网络请求或数值库。必须重启内核后从上至下全部执行。所有主实验代码在本文内，不导入experiment.py。

版本记录与实际解释器环境分别核对。若需检查解释器位置，临时运行`print(sys.executable)`；分享前删除个人路径输出。

## 1 解释器版本
import载入标准库sys；点号取模块中的版本文本。此处不显示个人文件路径。

In [1]:
# 1. Record the interpreter version without exposing a personal file path.
import sys
print("Python", sys.version)

Python 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]


## 先手算一个预测
先乘后加。print明确显示结果；交互式自动显示不等于脚本会打印。这里的7是预测，标签仍是8。

In [2]:
print(2 * 3 + 1)
print("third training label", 8)

7
third training label 8


## 变量按执行次序更新
赋值先算右侧。改变x并不会自动更新prediction；第三个print才显示重算后的值。

In [3]:
x = 3
bias = 1
prediction = 2 * x + bias
x = 4
print(prediction)
prediction = 2 * x + bias
print(prediction)

7
9


## 数字 文本与真假
type检查类型，双等号比较。这里只运行有定义的表达式；故意类型错误留在独立error_examples目录。

In [4]:
print(type(3), type(3.0), type("3"))
print(2 * 3, 2 * "3")
print(1 / 5, 7 == 8, 0.2 < 0.8)

<class 'int'> <class 'float'> <class 'str'>
6 33
0.2 False True


## 阶段 2
两个平行列表必须按同一记录顺序对齐。索引从0开始，第三条索引为2。候选升序且预先固定。

In [5]:
# 2. The same five calibration records as Unit 001.
train_x = [1, 2, 3, 4, 5]
train_y = [3, 5, 8, 9, 11]
candidates = [0, 1, 2]

### 索引与累计值
range(5)产生0到4；abs取绝对值；total在循环前初始化。每轮打印索引、误差和累计值，保留第三行偏差。

In [6]:
print(train_x[2], train_y[2])
total = 0
for i in range(len(train_y)):
    prediction = 2 * train_x[i] + 1
    error = abs(prediction - train_y[i])
    total = total + error
    print(i, error, total)
print("manual-loop MAE", total / len(train_y))

3 8
0 0 0
1 0 0
2 1 1
3 0 1
4 0 1
manual-loop MAE 0.2


## 阶段 3
def定义函数，调用才执行。return把结果交回；缩进决定何时返回。mae只检查非空和等长，有限数值、单位和对应顺序由本数据保证。raise ValueError明确拒绝不合格长度；详细异常处理留给003。

In [7]:
# 3. Define the two prediction rules and the average absolute error.
def predict_a(x, bias):
    return 2 * x + bias


def mae(actual, predicted):
    n = len(actual)
    if n == 0 or len(predicted) != n:
        raise ValueError("Need nonempty equal-length lists")
    total = 0
    for i in range(n):
        total = total + abs(predicted[i] - actual[i])
    return total / n


def predict_b(known_x, known_y, x):
    for i in range(len(known_x)):
        if x == known_x[i]:
            return known_y[i]
    return 0

## 阶段 4
外层换候选，内层处理五条训练输入。每个候选都创建新的预测列表。只在分数严格降低时更新，升序保证同分保留较小值。这里没有使用测试标签。

In [8]:
# 4. Select the bias using training records only.
# Candidates are increasing, and a strict comparison keeps the smaller tie.
best_bias = candidates[0]
initial_predictions = []
for x in train_x:
    initial_predictions.append(predict_a(x, best_bias))
best_score = mae(train_y, initial_predictions)
candidate_scores = []
for bias in candidates:
    predicted = []
    for x in train_x:
        predicted.append(predict_a(x, bias))
    score = mae(train_y, predicted)
    candidate_scores.append(score)
    print("candidate", bias, "train MAE", score)
    if score < best_score:
        best_score = score
        best_bias = bias

candidate 0 train MAE 1.2
candidate 1 train MAE 0.2
candidate 2 train MAE 0.8


## 阶段 5
先记录训练误差，然后对四个测试输入生成并显示预测。B检查完所有训练行仍未命中才返回0。此处冻结偏移量及预测。

In [9]:
# 5. Record training errors and freeze the test predictions.
a_train = []
b_train = []
for x in train_x:
    a_train.append(predict_a(x, best_bias))
    b_train.append(predict_b(train_x, train_y, x))
a_train_mae = mae(train_y, a_train)
b_train_mae = mae(train_y, b_train)
print("A train MAE", a_train_mae)
print("B train MAE", b_train_mae)

test_x = [1.5, 2.5, 3.5, 4.5]
a_test = []
b_test = []
for x in test_x:
    a_test.append(predict_a(x, best_bias))
    b_test.append(predict_b(train_x, train_y, x))
print("frozen bias", best_bias)
print("A predictions", a_test)
print("B predictions", b_test)

A train MAE 0.2
B train MAE 0.0
frozen bias 1
A predictions [4.0, 6.0, 8.0, 10.0]
B predictions [0, 0, 0, 0]


## 阶段 6
现在揭示公开教学标签，不能再回头调参数。公开代码不是安全封存。先手算总误差0与28，再看程序的平均。

In [10]:
# 6. Only now reveal the public teaching answers. Do not retune the bias.
test_y = [4, 6, 8, 10]
a_test_mae = mae(test_y, a_test)
b_test_mae = mae(test_y, b_test)
print("A test MAE", a_test_mae)
print("B test MAE", b_test_mae)

A test MAE 0.0
B test MAE 7.0


## 阶段 7
压力关系换成3x+1，使用旧预测，不能重训。print在未闭合圆括号里换行仍是同一次调用。绝对误差总和12，平均3。

In [11]:
# 7. The changed relation is 3*x + 1, as in Unit 001.
stress_y = [5.5, 8.5, 11.5, 14.5]
a_stress_mae = mae(stress_y, a_test)
print("A stress MAE", a_stress_mae)
print("test x | label | A | B | stress label | A stress error")
for i in range(len(test_x)):
    print(test_x[i], test_y[i], a_test[i], b_test[i],
          stress_y[i], abs(a_test[i] - stress_y[i]))

A stress MAE 3.0
test x | label | A | B | stress label | A stress error
1.5 4 4.0 0 5.5 1.5
2.5 6 6.0 0 8.5 2.5
3.5 8 8.0 0 11.5 3.5
4.5 10 10.0 0 14.5 4.5


## 一个会暴露误差抵消的检查
标签8与8，预测7与9，两个绝对误差均为1。打印布尔比较方便核对，这还不是后续单元的完整自动测试框架。

In [12]:
print("two-error MAE", mae([8, 8], [7, 9]))
print("candidate agreement", candidate_scores == [1.2, 0.2, 0.8])
print("selected bias agreement", best_bias == 1)
print("test agreement", a_test_mae == 0 and b_test_mae == 7)
print("stress agreement", a_stress_mae == 3)

two-error MAE 1.0
candidate agreement True
selected bias agreement True
test agreement True
stress agreement True


## 结束前的复现检查
重启内核并运行全部单元格，确认没有异常且核对结果均为True。关闭交互会话后，再从终端执行experiment.py。错误案例在error_examples目录单独运行，预期故意报错不能混作主实验成功。

一句限定范围的结论：五条训练记录选出偏移量1；四个合成测试点上A的MAE为0、B为7；关系改变后A的MAE为3。这不证明任何真实设备或深度网络的表现。

作者构建端使用全新Python进程中的真实IPython进程内内核顺序执行，不声称测试了本地Jupyter浏览器界面或跨操作系统安装。